# 🎚️ WE4 · Notebook 3b — Tuning the Critic's Verdict
## How far ahead should we look before we trust the critic?

> **This notebook picks up exactly where notebook 03 stopped.** Same company, same three-day
> campaign, same code. You already have an **actor**, a **critic**, and a working **A2C** loop, and
> you have seen it recover the retention team's sheet without ever waiting for a campaign to finish.
>
> Nothing about the world changes today. What changes is **one number inside the advantage**.

In notebook 03 the actor's weight on every move was the **one-step** TD error:

$$\delta_t \;=\; r_t + \gamma V(s_{t+1}) - V(s_t)$$

We adopted that because it removed the wait for the campaign to end — and it does. But look at what
it costs: of everything that happens after a move, we keep exactly **one real day** of evidence and
hand the entire rest of the story to the critic's guess. In notebook 02, REINFORCE did the opposite:
it kept **every** real day and used no guess at all.

Those are not two algorithms. They are **the two ends of one dial**, and nobody made us sit at either
end.

**What we are going to do**
1. Put a number on what the one-step choice actually costs, by **measuring** the two ingredients that
   matter — how noisy each estimator is, and how much it leans on the critic.
2. Build the **n-step advantage**: keep `n` real days, then let the critic take over. Notebook 03 is
   `n=1`; notebook 02 is `n=∞`.
3. Run A2C at several settings of the dial on the same world, same budget, and read the result
   honestly — including where the dial does *not* help.
4. *(Optional, at the end)* meet **GAE**, which stops choosing an `n` at all.

**How this notebook works**
- Same as before: short explanations, small hands-on tasks marked **🎯**, and a toy world small enough
  to check every estimate against the exact answer.
- 💰 All money is expected ad revenue per learner, in **CHF**.

> 🧭 **What this notebook assumes.** Only that you have *understood* notebook 03 — V, Q, the
> advantage, TD learning, the critic, and the entropy bonus are taken as known and are not
> re-explained here. It does **not** depend on notebook 03's code: this file is standalone, defines
> its own world, and runs on its own.

## 0. Setup

**This notebook is fully self-contained.** It clones nothing and imports no course helper module —
the world and every function it needs are defined below, in two cells. Run them and move on.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import itertools
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"]  = True
plt.rcParams["grid.alpha"] = 0.3

print("Environment ready ✅  ·  torch", torch.__version__)

### 0.1 · The world, and the pieces you already know

Two cells. The first is the campaign itself — three small tables. The second is the standard
machinery from notebooks 02 and 03: the episode sampler, the critic network, the two losses, the
entropy, and the exact-value solver we can afford because the world is tiny.

**Nothing here is new**, and nothing here is a task. Run both cells and move on — Part 1 is where
today starts. (Everything is written out in full so this notebook stands alone; if a line looks
unfamiliar, it is notebook 02 §1 or notebook 03 §4.)

In [ ]:
#@title 🔧 The world, and the code you already know (notebooks 02 & 03)  { display-mode: "form" }
# ── The campaign, as three small tables ──────────────────────────────────────
# These are the same rules you used in notebooks 02 and 03: what each move books today,
# and where it leaves the learner tomorrow.
ENGAGE  = ["😴 Cold", "🙂 Warm", "🔥 Hot"]        # engagement levels (the state)
ACTIONS = ["⏸️ Wait", "🔔 Nudge", "📺 Ad blast"]  # the three moves

# REWARD[state][action] — expected ad margin booked that day, in CHF
REWARD = [[0.0, -0.3, 0.2],       # 😴 Cold: a nudge costs, an ad blast on a cold learner earns little
          [0.2,  0.5, 2.0],       # 🙂 Warm
          [1.0,  1.5, 6.0]]       # 🔥 Hot : this is where the money is

# TRANS[state][action] — [(next state, probability), ...]
TRANS = [
    [ [(0, 0.9), (1, 0.1)], [(1, 0.6), (0, 0.4)], [(0, 0.9), (1, 0.1)] ],   # 😴 Cold
    [ [(1, 0.5), (0, 0.5)], [(2, 0.7), (1, 0.3)], [(1, 0.5), (0, 0.5)] ],   # 🙂 Warm
    [ [(2, 0.6), (1, 0.4)], [(2, 0.8), (1, 0.2)], [(1, 0.7), (0, 0.3)] ],   # 🔥 Hot
]

START_PROBS = [0.5, 0.5, 0.0]     # who walks in: nobody starts 🔥 Hot
N_DAYS      = 3
GAMMA       = 0.9                 # tomorrow's franc is worth 90% of today's

def transition(state, action):
    reward   = REWARD[state][action]
    outcomes = TRANS[state][action]
    next_state = np.random.choice([s for s, p in outcomes], p=[p for s, p in outcomes])
    return int(next_state), float(reward)

def returns_to_go(rewards, gamma=GAMMA):
    out = [0.0] * len(rewards)
    for t in reversed(range(len(rewards))):
        out[t] = rewards[t] + gamma * (out[t + 1] if t + 1 < len(rewards) else 0.0)
    return out

def action_probs(theta, state):
    return torch.softmax(theta[state], dim=-1)

def sample_episode(theta):
    engagement = int(np.random.choice(len(ENGAGE), p=START_PROBS))
    states, actions, rewards, next_states, log_probs = [], [], [], [], []
    for day in range(N_DAYS):
        p = action_probs(theta, engagement)
        action = int(torch.multinomial(p, 1))
        log_probs.append(torch.log(p[action]))
        next_engagement, reward = transition(engagement, action)
        states.append(engagement); actions.append(action); rewards.append(reward)
        next_states.append(next_engagement)
        engagement = next_engagement
    return states, actions, rewards, next_states, log_probs

def collect(theta, batch_size):
    '''A batch of campaigns, flattened into one pile of transitions.'''
    days, states, rewards, next_states, log_probs = [], [], [], [], []
    for _ in range(batch_size):
        s, a, r, s_next, lp = sample_episode(theta)
        days        += list(range(N_DAYS))
        states      += s
        rewards     += r
        next_states += s_next
        log_probs   += lp
    return (torch.tensor(days), torch.tensor(states), torch.tensor(rewards, dtype=torch.float32),
            torch.tensor(next_states), torch.stack(log_probs))

class Critic(nn.Module):
    def __init__(self, hidden=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(N_DAYS + len(ENGAGE), hidden), nn.Tanh(),
                                 nn.Linear(hidden, 1))
    def forward(self, x):
        return self.net(x).squeeze(-1)

def features(days, states):
    days, states = torch.as_tensor(days), torch.as_tensor(states)
    return torch.cat([F.one_hot(days.clamp(max=N_DAYS - 1), N_DAYS),
                      F.one_hot(states, len(ENGAGE))], dim=-1).float()

def V_hat(critic, days, states):
    days = torch.as_tensor(days)
    v = critic(features(days, states))
    return torch.where(days >= N_DAYS, torch.zeros_like(v), v)

def critic_loss(v_pred, target):
    return ((v_pred - target) ** 2).mean()

def policy_entropy(theta, states):
    p = torch.softmax(theta[states], dim=-1)
    return -(p * torch.log(p)).sum(dim=-1).mean()

def exact_values(theta, gamma=GAMMA):
    V = np.zeros((N_DAYS + 1, len(ENGAGE)))
    Q = np.zeros((N_DAYS, len(ENGAGE), len(ACTIONS)))
    for t in reversed(range(N_DAYS)):
        for s in range(len(ENGAGE)):
            pi = action_probs(theta, s).detach().numpy()
            for a in range(len(ACTIONS)):
                tomorrow = sum(p * V[t + 1][ns] for ns, p in TRANS[s][a])
                Q[t, s, a] = REWARD[s][a] + gamma * tomorrow
            V[t, s] = sum(pi[a] * Q[t, s, a] for a in range(len(ACTIONS)))
    return V[:N_DAYS], Q

def exact_J(theta, gamma=GAMMA):
    V, _ = exact_values(theta, gamma)
    return float(np.dot(START_PROBS, V[0]))

def sheet_J(sheet, gamma=GAMMA):
    logits = torch.full((len(ENGAGE), len(ACTIONS)), -20.0)
    for s, a in enumerate(sheet):
        logits[s, a] = 20.0
    return exact_J(logits, gamma)

def value_of(V, day, state):
    '''V(day, state), with a finished campaign worth 0.'''
    return 0.0 if day >= N_DAYS else float(V[day][state])

J_UNIFORM = exact_J(torch.zeros(len(ENGAGE), len(ACTIONS)))
J_BEST    = max(sheet_J(list(sh)) for sh in itertools.product(range(len(ACTIONS)),
                                                              repeat=len(ENGAGE)))

# The frozen, opinionated policy from notebook 03 §2 — our measuring bench for Part 1.
theta_pi = torch.tensor([[0.0,  0.8, -0.5],
                         [0.0,  0.8, -0.5],
                         [0.0, -0.5,  0.8]])
V_PI, Q_PI = exact_values(theta_pi)

print(f"J(uniform) = {J_UNIFORM:+.3f}   ·   J(best sheet) = {J_BEST:+.3f}")
print("World and machinery ready ✅  — this notebook needs nothing else.")

---
# Part 1 — One dial, two ends you have already used

Line up the two weights you have written in this course, for the same move on the same morning:

$$
\textbf{notebook 02:}\quad G_t - V(s_t)
\qquad\qquad
\textbf{notebook 03:}\quad r_t + \gamma V(s_{t+1}) - V(s_t)
$$

They differ in exactly one respect: **how many real days of evidence you keep before you let the
critic finish the sentence.**

- Notebook 02 keeps **all** of them. No guess enters the number — but every roll of the dice for the
  rest of the campaign is inside it, so the number is **noisy**.
- Notebook 03 keeps **one**. Only one day's dice can move it, so it is **calm** — but everything from
  day 2 onwards is the critic's opinion, and early in training the critic is wrong.

So the choice is not *"real rewards or a critic"*. It is **how much of each**, and it has a name:

| | keep `n` real days | then |
|---|---|---|
| `n = 1` | today only | trust the critic for the rest — *notebook 03* |
| `n = 2` | today and tomorrow | trust the critic for the rest |
| `n = 3` (`= ∞` here) | the whole campaign | nothing left to trust — *notebook 02* |

$$
G^{(n)}_t \;=\; r_t + \gamma r_{t+1} + \dots + \gamma^{\,n-1} r_{t+n-1}
\;+\; \underbrace{\gamma^{\,n} V(s_{t+n})}_{\text{the critic finishes it}}
\qquad
A^{(n)}_t \;=\; G^{(n)}_t - V(s_t)
$$

> 🧭 Our campaign is only three days long, so `n = 3` already *is* the full return. That is a feature
> for teaching: the whole dial fits on one screen and we can compute every point on it exactly.

### 🎯 Task 1 — the n-step target

One blank. Build `G` by walking forward at most `n` days from day `t`, discounting each day as you go,
then — if the campaign has **not** ended by then — add the critic's opinion of where you stopped,
discounted by `gamma ** steps`. That second half is written for you.

> 💡 The scaffolding already works out `steps` (how many real days are actually available: near the
> end of a campaign there may be fewer than `n`) and `ep`, the index at which this episode's day 0
> sits in the flattened batch. So day `t + k` of this episode is at position `ep + t + k`.

In [ ]:
def n_step_target(critic, days, rewards, next_states, n, gamma=GAMMA):
    '''n-step targets for a flat batch of transitions (blocks of N_DAYS per campaign).'''
    T = len(rewards)
    targets = torch.zeros(T)
    with torch.no_grad():                      # a target is a label, exactly as in notebook 03
        for i in range(T):
            t     = int(days[i])               # which morning of ITS campaign this step is
            ep    = i - t                      # where that campaign's day 0 sits in the batch
            steps = min(n, N_DAYS - t)         # real days actually available from here
            G = ______      # discounted sum of the next `steps` real rewards, from rewards[ep + t]
            if t + steps < N_DAYS:             # the campaign is still running → the critic finishes it
                last = ep + t + steps - 1
                G = G + gamma**steps * float(V_hat(critic, [t + steps],
                                                   [int(next_states[last])]))
            targets[i] = G
    return targets

# --- self-check on one batch, against two things we can compute independently
torch.manual_seed(0); np.random.seed(0)
_critic = Critic()
_days, _states, _rewards, _next, _lp = collect(theta_pi, 4)

# (a) n = 1 must reproduce notebook 03's TD target exactly
with torch.no_grad():
    td1 = _rewards + GAMMA * V_hat(_critic, _days + 1, _next)
assert torch.allclose(n_step_target(_critic, _days, _rewards, _next, n=1), td1, atol=1e-5), \
    "n=1 should be exactly notebook 03's r + γ·V̂(s′)."

# (b) n = N_DAYS must reproduce the plain discounted return — no critic left in it
big = n_step_target(_critic, _days, _rewards, _next, n=N_DAYS)
mc  = torch.tensor(np.concatenate([returns_to_go(_rewards[i:i+N_DAYS].tolist(), GAMMA)
                                   for i in range(0, len(_rewards), N_DAYS)]), dtype=torch.float32)
assert torch.allclose(big, mc, atol=1e-5), "n=3 should be the full return-to-go, with no bootstrap."
print("✅ n=1 is notebook 03's target, and n=3 is notebook 02's return. Same function, one dial.")

### 🎯 Task 2 — the same campaign, judged at both ends of the dial

Before we put the dial in the training loop, work it by hand on **one** campaign, using the exact
`V_PI` so there is no critic to blame. For the move made on day `t` you need the advantage at each
setting of `n` — and the advantage is always **the n-step return minus the value of where we were**:

$$A^{(n)}_t = G^{(n)}_t - V(s_t)$$

One blank: the `n`-step return `G` is already built for you in `g`, so the advantage is one
subtraction away. Use `value_of(V_PI, t, states[t])`.

In [ ]:
np.random.seed(11); torch.manual_seed(11)
states, actions, rewards, next_states, _ = sample_episode(theta_pi)

print("one campaign:", " → ".join(ENGAGE[s] for s in states))
print("moves       :", " , ".join(ACTIONS[a] for a in actions))
print("booked      :", [round(r, 2) for r in rewards], "CHF\n")

print(f"{'day':>4s}{'n=1 (nb 03)':>14s}{'n=2':>10s}{'n=3 (nb 02)':>14s}{'exact A':>11s}")
for t in range(N_DAYS):
    row = []
    for n in [1, 2, 3]:
        steps = min(n, N_DAYS - t)
        g = sum(GAMMA**k * rewards[t + k] for k in range(steps))          # the n-step return
        if t + steps < N_DAYS:
            g += GAMMA**steps * value_of(V_PI, t + steps, next_states[t + steps - 1])
        advantage = ______        # the n-step return minus the value of where we were
        row.append(advantage)
    exact = Q_PI[t, states[t], actions[t]] - V_PI[t, states[t]]
    print(f"{t:>4d}{row[0]:>14.3f}{row[1]:>10.3f}{row[2]:>14.3f}{exact:>11.3f}")

assert abs(row[2] - (rewards[N_DAYS-1] - value_of(V_PI, N_DAYS-1, states[N_DAYS-1]))) < 1e-9, \
    "On the last morning every n collapses to the same thing: there is no tomorrow to add."
print("\n✅ On the last day all three columns agree — nothing is left to disagree about.")
print("   Earlier days differ: each setting keeps a different amount of the real story.")

Read those two assertions again — they are the point of the whole notebook. **One function**, and at
its two extreme settings it becomes the two algorithms you already wrote. Everything between them is
new territory that neither notebook visited.

## 1.1 · What the dial actually trades

Words like *"noisier"* and *"leans on the critic"* are cheap. Let's put numbers on both, on the same
transitions, the way we measured the baseline in notebook 02.

The setup is the toy-world luxury one last time: freeze the policy `theta_pi`, hand every estimator
the **exact** `V` (so no estimator is handicapped by a bad critic), and let them all score the **same**
4000 campaigns. Any difference in spread is then purely the estimator's own doing.

### 🎯 Task 3 — the four weights, on identical data

Three blanks, all of them the formulas from the table above written once each. For the step at day
`t` of one campaign:

- `w_reinforce` — the plain return-to-go `G[t]`, with **no** baseline (notebook 02, before we fixed it)
- `w_baseline` — the same return, now **minus** the value of where we were: `G[t] − V(t, s)`
- `w_a2c` — the **one-step** TD error: `r_t + γ·V(t+1, s′) − V(t, s)`

Use `value_of(V_PI, day, state)` for every `V` lookup — it already returns 0 for a finished campaign.

In [ ]:
def estimator_weights(n_campaigns=4000, seed=1):
    '''Score the SAME campaigns four ways. Returns a dict of flat arrays.'''
    np.random.seed(seed); torch.manual_seed(seed)
    out = {"reinforce": [], "baseline": [], "2-step": [], "a2c": []}
    for _ in range(n_campaigns):
        states, actions, rewards, next_states, _ = sample_episode(theta_pi)
        G = returns_to_go(rewards, GAMMA)
        for t in range(N_DAYS):
            s, s_next = states[t], next_states[t]
            w_reinforce = ______      # the return-to-go, with no baseline at all
            w_baseline  = ______      # that same return, minus the value of where we were
            w_a2c       = ______      # one real day, then the critic: r + γ·V(s′) − V(s)
            # the 2-step weight is given, so you can see the pattern the dial follows
            if t + 1 < N_DAYS:
                g2 = rewards[t] + GAMMA * rewards[t+1] + GAMMA**2 * value_of(V_PI, t+2, next_states[t+1])
            else:
                g2 = rewards[t]
            w_2step = g2 - value_of(V_PI, t, s)

            out["reinforce"].append(w_reinforce); out["baseline"].append(w_baseline)
            out["2-step"].append(w_2step);        out["a2c"].append(w_a2c)
    return {k: np.array(v) for k, v in out.items()}

W = estimator_weights()

print(f"{'weight put on log π':34s}{'mean':>9s}{'variance':>11s}{'std':>8s}")
for k, name in [("reinforce", "G  (no baseline)  · nb 02"),
                ("baseline",  "G − V   (n = 3 = full)"),
                ("2-step",    "2-step advantage"),
                ("a2c",       "1-step TD error δ · nb 03")]:
    a = W[k]
    print(f"{name:34s}{a.mean():9.3f}{a.var():11.3f}{a.std():8.3f}")

assert W["reinforce"].var() > W["baseline"].var() > W["a2c"].var(), \
    "Variance should fall as we add a baseline and then shorten the horizon."
print("\n✅ Variance falls monotonically as the dial turns towards the critic.")

### 🔬 Read that table slowly

Two separate things happened, and confusing them is the classic mistake:

**1 · The baseline re-centred the number.** The top row averages around **+1.9**; every row below it
averages around **0**. Without a baseline, *every* move on a profitable campaign gets pushed up —
including the bad ones — just less enthusiastically than the good ones. Subtracting `V(s)` is what
turns the weight from *"was this campaign profitable?"* into *"was this move better than my habit?"*
That is a change of **meaning**, not just of spread. (This is notebook 02's fix, seen from a new angle.)

**2 · Shortening the horizon calmed it down.** Rows two, three and four all mean the same thing —
they are all estimates of the same advantage — but their spread keeps shrinking as we hand more of
the future to the critic. Each real day we drop is a day of dice that can no longer shake the number.

Here is the same table as a picture.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14.5, 4.2))
order  = ["reinforce", "baseline", "2-step", "a2c"]
labels = ["G\n(no baseline)", "G − V\n(n=3, nb 02)", "2-step\nadvantage", "δ = 1-step\n(nb 03)"]
colors = ["#dd8452", "#c0a15b", "#7a9bb8", "#4a5bd0"]

ax[0].boxplot([W[k] for k in order], showfliers=False, tick_labels=labels)
ax[0].axhline(0, color="#444", lw=1)
ax[0].set_ylabel("weight put on log π(a|s)")
ax[0].set_title("Same 4000 campaigns, four ways to score them")

for k, c, lab in zip(order, colors, ["G (no baseline)", "G − V (n=3)", "2-step", "δ (n=1)"]):
    ax[1].hist(W[k], bins=60, alpha=0.45, color=c, label=lab, density=True)
ax[1].axvline(0, color="#444", lw=1)
ax[1].set_xlabel("weight"); ax[1].set_ylabel("density")
ax[1].set_title("Same data, four signals"); ax[1].legend(fontsize=7)

ax[2].bar(range(4), [W[k].var() for k in order], color=colors)
ax[2].set_xticks(range(4)); ax[2].set_xticklabels(labels, fontsize=8)
ax[2].set_ylabel("variance of the weight")
ax[2].set_title("Turning the dial towards the critic lowers the noise")
plt.tight_layout(); plt.show()

print("Left  : the top box is off-centre — no baseline. The other three are centred on 0.")
print("Middle: the orange pile sits to the RIGHT of zero; the blue one is tallest and narrowest.")
print("Right : among the three centred estimators, variance falls as n shrinks.")

> ⚠️ **And the catch, which this measurement cannot show you.** Every estimator above used the
> **exact** `V`. In a real run the critic is *learned*, and a wrong `V` leaks straight into a short-`n`
> advantage — with `n=1`, two-thirds of the number is the critic's opinion. The full return has no
> such leak: it is noisy, but it is **not wrong on average**, whatever the critic believes.
>
> That is the actual trade: **short `n` buys calm and pays in bias; long `n` buys honesty and pays in
> noise.** The measurement above only priced one side of it. The other side needs a training run.

---
# Part 2 — n-step A2C

Now we spend the dial where it counts: inside the training loop.

Nothing about A2C's *shape* changes. It is still notebook 03's five steps — collect, ask the critic,
build one loss out of three terms, one step on each network, discard the batch. The **only** edit is
which target the critic aims at and which advantage the actor is handed.

### 🎯 Task 4 — swap the one-step target for the dial

Two blanks in the middle of notebook 03's loop.

- **(a)** the `target`, from your `n_step_target` — the critic aims at it, and the actor's advantage
  is built from it.
- **(b)** the `advantage` — target minus what the critic expected. Remember the `.detach()` rule from
  notebook 03 §6.2: the actor is handed a **verdict**, not a variable it may edit.

In [ ]:
def train_a2c_nstep(n=1, n_iters=300, batch_size=24, lr_actor=0.05, lr_critic=0.05,
                    ent_coef=0.01, value_coef=0.5, gamma=GAMMA, seed=0, log_every=0):
    '''Notebook 03's A2C, with the horizon of the advantage as a parameter.'''
    torch.manual_seed(seed); np.random.seed(seed)
    theta  = torch.zeros(len(ENGAGE), len(ACTIONS), requires_grad=True)
    critic = Critic()
    opt_actor  = torch.optim.Adam([theta],             lr=lr_actor)
    opt_critic = torch.optim.Adam(critic.parameters(), lr=lr_critic)
    history, critic_error, entropy_hist, policy_hist = [], [], [], []

    for it in range(n_iters):
        # 1 · COLLECT — unchanged from notebook 03
        days, states, rewards, next_states, log_probs = collect(theta, batch_size)

        # 2 · ASK THE CRITIC — the only lines that changed
        v_now = V_hat(critic, days, states)
        target    = ______      # the n-step target for this batch (your Task 1 function)
        advantage = ______      # target vs what the critic expected — a verdict, so detach v_now

        # 3 · THREE TERMS, ONE LOSS — unchanged from notebook 03
        actor_loss = -(advantage * log_probs).mean()
        entropy    = policy_entropy(theta, states)
        loss = actor_loss + value_coef * critic_loss(v_now, target) - ent_coef * entropy

        # 4 · ONE STEP on each network
        opt_actor.zero_grad(); opt_critic.zero_grad()
        loss.backward()
        opt_actor.step(); opt_critic.step()

        # 5 · scorekeeping (toy-world luxury)
        V_exact, _ = exact_values(theta, gamma)
        history.append(float(np.dot(START_PROBS, V_exact[0])))
        with torch.no_grad():
            V_now = np.array([[float(V_hat(critic, [t], [s])) for s in range(len(ENGAGE))]
                              for t in range(N_DAYS)])
        critic_error.append(float(np.abs(V_now - V_exact).mean()))
        entropy_hist.append(float(entropy.detach()))
        policy_hist.append(torch.softmax(theta.detach(), dim=-1).numpy().copy())
        if log_every and (it + 1) % log_every == 0:
            print(f"  iter {it+1:4d}   J(θ) = {history[-1]:+.3f}   critic error = {critic_error[-1]:.3f}")

    return {"theta": theta.detach(), "critic": critic, "J": history,
            "critic_error": critic_error, "entropy": entropy_hist,
            "policy": np.array(policy_hist)}

# --- self-check: n=1 must reproduce notebook 03's A2C exactly (same seed, same numbers)
out1 = train_a2c_nstep(n=1, n_iters=20, seed=0); h1 = out1["J"]
assert h1[0] < h1[-1], "Twenty iterations should already move J upwards."
print(f"✅ n=1 runs: J went {h1[0]:+.3f} → {h1[-1]:+.3f} in 20 iterations (this IS notebook 03's A2C).")

### Turn the dial and watch

Three runs, same world, same seed, same budget of campaigns — only `n` differs. `n=1` is notebook 03;
`n=3` uses the whole campaign and is REINFORCE-with-a-learned-critic-as-baseline.

In [ ]:
runs = {}
for n in [1, 2, 3]:
    print(f"training with n = {n}…")
    runs[n] = train_a2c_nstep(n=n, seed=0)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
colors = {1: "#4a5bd0", 2: "#579b6a", 3: "#dd8452"}
for n in [1, 2, 3]:
    ax[0].plot(runs[n]["J"], color=colors[n], lw=1.8,
               label=f"n = {n}" + ("  (notebook 03)" if n == 1 else "  (full campaign)" if n == 3 else ""))
    ax[1].plot(runs[n]["critic_error"], color=colors[n], lw=1.8, label=f"n = {n}")
ax[0].axhline(J_BEST,    ls="--", color="#444", lw=1, label="best possible sheet")
ax[0].axhline(J_UNIFORM, ls=":",  color="#999", lw=1, label="uniform policy")
ax[0].set_xlabel("iteration (one batch of 24 campaigns)"); ax[0].set_ylabel("J(θ) — exact")
ax[0].set_title("Did the actor learn?"); ax[0].legend(fontsize=8)
ax[1].set_xlabel("iteration"); ax[1].set_ylabel("mean |V̂ − V^π|")
ax[1].set_title("Did the critic keep up?"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"{'setting':22s}{'J after 300 batches':>21s}{'batches to reach J = 3.5':>26s}")
for n in [1, 2, 3]:
    hit = next((i for i, x in enumerate(runs[n]["J"]) if x > 3.5), None)
    print(f"{'n = ' + str(n):22s}{np.mean(runs[n]['J'][-20:]):>21.3f}{(hit if hit is not None else '—'):>26}")
print(f"{'best possible sheet':22s}{J_BEST:>21.3f}")

### 🎬 Watch the sheet get written

A learning curve tells you *that* it worked. This tells you *what it decided*, and when.

Below is the actor's policy `π(a|s)` at every iteration — one panel per engagement level, one line
per move. Every panel starts at `1/3, 1/3, 1/3` (the uniform policy, no opinions) and you can watch
the retention sheet resolve itself line by line.

In [ ]:
run = runs[2]                                   # the n = 2 run from above
P   = run["policy"]                             # (iterations, states, actions)
act_colors = ["#999999", "#579b6a", "#4a5bd0"]

fig, ax = plt.subplots(1, len(ENGAGE), figsize=(13, 3.6), sharey=True)
for s in range(len(ENGAGE)):
    for a in range(len(ACTIONS)):
        ax[s].plot(P[:, s, a], color=act_colors[a], lw=2, label=ACTIONS[a])
    ax[s].axhline(1/3, ls=":", color="#bbb", lw=1)
    ax[s].set_title(ENGAGE[s]); ax[s].set_xlabel("iteration"); ax[s].set_ylim(-0.02, 1.02)
ax[0].set_ylabel("π(a | s)"); ax[0].legend(fontsize=8)
fig.suptitle("The actor writing the sheet, one line per engagement level")
plt.tight_layout(); plt.show()

print("Final sheet read off the actor:")
for s in range(len(ENGAGE)):
    a = int(np.argmax(P[-1, s]))
    print(f"   {ENGAGE[s]:10s} → {ACTIONS[a]:12s}  (π = {P[-1, s, a]:.2f})")

> 👀 **Two things worth noticing.** The 🔥 Hot line resolves almost immediately — cashing in on an
> engaged learner pays today, so the advantage says so from the first batch. The 😴 Cold and 🙂 Warm
> lines take longer, because their best move pays *tomorrow*, and the actor can only see that once
> the critic has learned what tomorrow is worth. **The order in which the sheet gets written is the
> critic becoming useful.**

### And how decided is it?

The same runs, seen through entropy — notebook 03's `H(π)`, tracked over training. It starts at
`log 3 ≈ 1.10` (maximally undecided) and falls as the sheet firms up. This is the curve the entropy
bonus is holding open.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11.5, 4))
for n in [1, 2, 3]:
    ax[0].plot(runs[n]["entropy"], color=colors[n], lw=1.8, label=f"n = {n}")
ax[0].axhline(float(np.log(3)), ls=":", color="#999", lw=1, label="log 3 — fully undecided")
ax[0].set_xlabel("iteration"); ax[0].set_ylabel("H(π)")
ax[0].set_title("How undecided the actor still is"); ax[0].legend(fontsize=8)

ax[1].plot(runs[2]["entropy"], color="#579b6a", lw=1.8)
ax[1].set_xlabel("iteration"); ax[1].set_ylabel("H(π)", color="#579b6a")
ax2 = ax[1].twinx(); ax2.grid(False)
ax2.plot(runs[2]["J"], color="#4a5bd0", lw=1.8)
ax2.set_ylabel("J(θ)", color="#4a5bd0")
ax[1].set_title("Certainty (green) and performance (blue), n = 2")
plt.tight_layout(); plt.show()

print(f"entropy at the start: {runs[2]['entropy'][0]:.3f}   (log 3 = {float(np.log(3)):.3f})")
print(f"entropy at the end  : {runs[2]['entropy'][-1]:.3f}   — the sheet is written")

> 🧭 **Why we are not sweeping the entropy coefficient here.** On a three-state world with three
> moves the actor cannot really trap itself — every state is visited constantly, so a move that was
> pushed down still gets sampled and can recover. Turn the bonus off in this world and almost nothing
> happens. That is a fact about the size of our toy, not a demotion of the idea: on a large action
> space the collapse notebook 03 warned about is real and permanent. Our world is too small to show
> the trap door, and rigging a seed to fake it would teach the wrong lesson.

### Does the critic actually predict?

One more diagnostic, and it is the one that tells you whether the apparatus is working. For a pile of
real transitions we compare the critic's `V̂(s)` against the **actual discounted money** that followed
from that morning. A critic that has learned sits on the diagonal.

In [ ]:
def value_vs_actual(critic, n_campaigns=400, seed=5):
    '''What the critic predicted, against what actually followed.'''
    np.random.seed(seed); torch.manual_seed(seed)
    pred, actual = [], []
    for _ in range(n_campaigns):
        st, ac, rw, nx, _ = sample_episode(theta_pi)
        G = returns_to_go(rw, GAMMA)
        with torch.no_grad():
            for t in range(N_DAYS):
                pred.append(float(V_hat(critic, [t], [st[t]])))
                actual.append(G[t])
    return np.array(pred), np.array(actual)

p_pre,  a_pre  = value_vs_actual(Critic())            # untrained
p_post, a_post = value_vs_actual(runs[2]["critic"])   # the critic from our n = 2 run

fig, ax = plt.subplots(1, 2, figsize=(11, 4.4), sharex=True, sharey=True)
for a, (pv, av, ttl) in zip(ax, [(p_pre, a_pre, "UNtrained critic"),
                                 (p_post, a_post, "TRAINED critic (n = 2 run)")]):
    a.scatter(av, pv, s=14, alpha=0.25, color="#4a5bd0")
    lim = [min(av.min(), pv.min()) - 0.5, max(av.max(), pv.max()) + 0.5]
    a.plot(lim, lim, "--", color="#cc3333", lw=1.5, label="perfect prediction")
    a.set_title(ttl); a.set_xlabel("money that actually followed (CHF)"); a.legend(fontsize=8)
ax[0].set_ylabel("critic's V̂(s)")
plt.tight_layout(); plt.show()

print(f"correlation with reality — untrained: {np.corrcoef(p_pre, a_pre)[0,1]:+.3f}"
      f"   ·   trained: {np.corrcoef(p_post, a_post)[0,1]:+.3f}")

### 🔬 Read that honestly, as in notebook 03

Notebook 03 asked *"was the critic worth it?"* and answered *"on this problem, not really — and the
reason is the lesson."* Same discipline here.

**What the plot does show.** All three settings end up in the same place — around `J = 3.81`, within
a whisker of the best possible sheet. The difference is **how fast they get there**: read the last
column of the table. `n = 1` needs noticeably more batches to cross `J = 3.5` than `n = 2` or `n = 3`,
and that ordering holds across seeds. The reason is the one we set up in Part 1: early in training the
critic is still bad, and `n = 1` is the setting that leans on it hardest. The longer horizons carry
more real money in each weight, so they are less hostage to a critic that has not learned yet.

Look at the right panel too — the critic's error behaves differently depending on which target it is
chasing, and it never fully settles, because the policy it is trying to evaluate keeps moving.

**What the plot cannot show, because our campaign is three days long.** The whole reason `n=1` exists
is horizon. At three steps, the "full return" survives only three rolls of the dice, so it is barely
noisier than the one-step version — the left end of the dial has almost nothing to buy. Put the same
dial on a thousand-step process and the picture inverts: the full return becomes hopeless while the
one-step estimate is barely affected. **Our toy is too small to punish `n=3`, and that is a fact about
our toy, not about RL.**

> 🧭 **The transferable rule.** `n` is not a tuning knob you sweep blindly; it is a statement about
> **how far you trust your critic relative to your horizon**. Long horizon and a decent critic → small
> `n`. Short horizon or a critic you do not trust yet → keep more real days.

## 2.1 · Watch the bias arrive

That last sentence deserves evidence rather than assertion, and we can build it in one cell.

We take a **deliberately biased critic** — one that has been trained, then had a constant added to
every prediction, so it is confidently wrong by a known amount — and ask each setting of the dial for
its advantage. Because `V` appears inside the bootstrap term, the error leaks into short-`n`
estimates and is *absent* from the full-return one.

In [ ]:
#@title 🔬 A critic that is wrong — where does its error end up?  { display-mode: "form" }
# We corrupt the exact V with a fixed, state-dependent error — which is what a half-trained critic
# actually looks like: not uniformly optimistic, just wrong in different directions in different
# cells. Then, for each (day, state, action), we AVERAGE many estimates and compare against the exact
# advantage Q − V. Averaging cancels the variance, so what is left on the plot is pure bias.

from collections import defaultdict

def advantage_bias(n, noise, n_campaigns=3000, seed=3):
    rng   = np.random.default_rng(0)
    V_bad = V_PI + rng.normal(0, noise, size=V_PI.shape)     # a systematically wrong critic
    np.random.seed(seed); torch.manual_seed(seed)
    seen = defaultdict(list)
    for _ in range(n_campaigns):
        st, ac, rw, nx, _ = sample_episode(theta_pi)
        for t in range(N_DAYS):
            steps = min(n, N_DAYS - t)
            g = sum(GAMMA**k * rw[t + k] for k in range(steps))
            if t + steps < N_DAYS:
                g += GAMMA**steps * value_of(V_bad, t + steps, nx[t + steps - 1])
            seen[(t, st[t], ac[t])].append(g - value_of(V_bad, t, st[t]))
    errs = [abs(np.mean(v) - (Q_PI[t, s, a] - V_PI[t, s]))
            for (t, s, a), v in seen.items() if len(v) >= 30]
    return float(np.mean(errs))

noises = [0.0, 0.5, 1.0, 2.0]
table  = {n: [advantage_bias(n, z) for z in noises] for n in [1, 2, 3]}

plt.figure(figsize=(7.5, 4.2))
for n in [1, 2, 3]:
    plt.plot(noises, table[n], "o-", color=colors[n], lw=2,
             label=f"n = {n}" + ("  (notebook 03)" if n == 1 else "  (no bootstrap in it)" if n == 3 else ""))
plt.xlabel("how wrong the critic is (std-dev of its per-cell error)")
plt.ylabel("bias of the reported advantage")
plt.title("A wrong critic contaminates short horizons most")
plt.legend(); plt.tight_layout(); plt.show()

for n in [1, 2, 3]:
    print(f"n = {n}:  bias at a perfect critic = {table[n][0]:.3f}   →   at error 2.0 = {table[n][-1]:.3f}")
assert table[1][-1] > table[3][-1], "Short horizons should suffer more from a wrong critic."
print("\n✅ With a badly wrong critic, n=1 is the most contaminated and n=3 the least.")

> 👀 **That is the bias half of the trade, made visible.** All three lines start at essentially zero
> — with a perfect critic every setting of the dial reports the true advantage, which is the sanity
> check. As the critic gets worse, they separate in the order the theory predicts: `n=1` is hurt most,
> because two-thirds of its number *is* the critic's opinion, and `n=3` least, because it contains no
> bootstrap at all — the critic enters it only as a baseline, and a baseline is allowed to be wrong
> (notebook 02 §4.2: any baseline that does not depend on the action leaves the gradient unbiased).
>
> Put the two measurements together and you have the entire content of the dial:
> **variance falls as `n` shrinks (Part 1); bias from a wrong critic grows as `n` shrinks (here).**
> Neither plot alone tells you where to sit. Together they do.

---
# Part 3 — The sheet, and the three algorithms side by side

The retention team still wants one sheet of paper. Let's check the dial did not cost us the answer,
and then put all three algorithms from this course on one plot.

In [ ]:
best = train_a2c_nstep(n=2, seed=0)
theta_best, hist_best = best["theta"], best["J"]
sheet_n2 = [int(torch.argmax(theta_best[s])) for s in range(len(ENGAGE))]

print("n-step A2C's sheet scores J =", round(sheet_J(sheet_n2), 3),
      " · the best possible sheet scores J =", round(J_BEST, 3))
assert abs(sheet_J(sheet_n2) - J_BEST) < 1e-6, "The dial should still find the optimal sheet."
print("✅ Same sheet as notebooks 02 and 03 — the dial changed the route, not the destination.")

### 📊 All three, same world, same budget

Notebook 02's REINFORCE (complete campaigns, batch-average baseline), notebook 03's A2C (`n=1`), and
today's dial at `n=2`. The REINFORCE run below is notebook 03's comparison code, unchanged.

In [ ]:
#@title 📊 Notebook 02's REINFORCE, re-run for comparison  { display-mode: "form" }
def train_reinforce(n_iters=300, batch_size=24, lr=0.05, gamma=GAMMA, seed=0):
    '''Notebook 02's algorithm: complete campaigns, per-state batch-average baseline.'''
    torch.manual_seed(seed); np.random.seed(seed)
    theta = torch.zeros(len(ENGAGE), len(ACTIONS), requires_grad=True)
    opt = torch.optim.Adam([theta], lr=lr)
    history = []
    for it in range(n_iters):
        batch = [sample_episode(theta) for _ in range(batch_size)]
        G = np.array([returns_to_go(r, gamma) for _, _, r, _, _ in batch])
        sums, counts = np.zeros(len(ENGAGE)), np.zeros(len(ENGAGE))
        for i, (states, _, _, _, _) in enumerate(batch):
            for t, s in enumerate(states):
                sums[s] += G[i, t]; counts[s] += 1
        baseline = np.where(counts > 0, sums / np.maximum(counts, 1), G.mean())
        loss = torch.zeros(())
        for i, (states, _, _, _, log_probs) in enumerate(batch):
            for t in range(N_DAYS):
                loss = loss - (G[i, t] - baseline[states[t]]) * log_probs[t]
        opt.zero_grad(); (loss / batch_size).backward(); opt.step()
        history.append(exact_J(theta, gamma))
    return theta.detach(), history

print("Re-running notebook 02's REINFORCE…")
_, hist_pg = train_reinforce()

plt.figure(figsize=(9.5, 4.8))
plt.plot(hist_pg,       color="#dd8452", lw=1.8, label="REINFORCE — batch-average baseline (nb 02)")
plt.plot(runs[1]["J"],  color="#4a5bd0", lw=1.8, label="A2C, n = 1 — learned critic (nb 03)")
plt.plot(hist_best,     color="#579b6a", lw=2.2, label="A2C, n = 2 — the dial (today)")
plt.axhline(J_BEST,    ls="--", color="#444", lw=1, label="best possible sheet")
plt.axhline(J_UNIFORM, ls=":",  color="#999", lw=1, label="uniform policy")
plt.xlabel("iteration (one batch of 24 campaigns)"); plt.ylabel("J(θ) — exact")
plt.title("Three notebooks, one world, one budget")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

print(f"{'algorithm':44s}{'J after 300 batches':>20s}")
for name, h in [("REINFORCE (nb 02)", hist_pg),
                ("A2C n=1 (nb 03)", runs[1]["J"]),
                ("A2C n=2 (today)", hist_best)]:
    print(f"{name:44s}{np.mean(h[-20:]):>20.3f}")
print(f"{'best possible sheet':44s}{J_BEST:>20.3f}")

> 🧭 **The same warning as notebook 03, and it still applies.** All three land on the same sheet,
> and on a three-state, three-day world the differences between them are small and partly luck of the
> seed. Do not read a winner off this plot. Read the **shapes**: REINFORCE needs a finished campaign,
> A2C needs none, and the dial lets you choose how much of the critic's opinion you are willing to
> buy. The first two facts are what survive contact with a real problem; the third is what you tune
> once you are there.

---
# ⭐ Optional — Do we have to pick an `n` at all?

> ### 🛑 The exercise stops here.
> **Everything below is optional and is not needed for anything that follows in this notebook.**
> **GAE has its own dedicated coding exercise in the next hour**, where it is built properly and used
> inside PPO. What follows is a two-cell preview for anyone who finished early and is curious about
> where the dial goes next — skipping it costs you nothing.

Everything above forced a choice: pick `n`, live with it. That should feel arbitrary, because it is.
`n=2` throws away everything the `n=1` and `n=3` estimates knew, and there is no reason the best
answer sits exactly on an integer.

**GAE (Generalized Advantage Estimation)** stops choosing. It takes an exponentially weighted average
of *every* `n` at once, controlled by one number `λ ∈ [0, 1]`, and — this is the part that makes it
practical — it collapses into a backwards recursion over the TD errors `δ` you already know:

$$
A^{\text{GAE}}_t \;=\; \delta_t \;+\; \gamma\lambda\, A^{\text{GAE}}_{t+1}
\qquad\text{where}\qquad
\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)
$$

And the two ends of the dial you spent this notebook on are its two extreme settings:

| `λ` | what it becomes | which notebook |
|---|---|---|
| `0` | exactly `δ_t`, the one-step advantage | notebook 03 |
| `1` | exactly the full-return advantage | notebook 02 |

Run the two cells below to see that claim checked numerically, and nothing more. The real treatment
is the next exercise.

> ### ⭐ 🎯 Optional Task 5 — the GAE recursion
> One blank, in the cell below, and only if you want it. The formula is the boxed one above:
> the current `delta`, plus the running total carried back from later days, discounted by `gamma*lam`.

In [ ]:
#@title ⭐ OPTIONAL — GAE in six lines, and the two ends of the dial  { display-mode: "form" }
def gae_advantages(rewards, states, next_states, V, gamma=GAMMA, lam=0.95):
    '''Backwards recursion over one campaign's TD errors.'''
    T = len(rewards)
    adv, running = np.zeros(T), 0.0
    for t in reversed(range(T)):
        delta   = rewards[t] + gamma * value_of(V, t + 1, next_states[t]) - value_of(V, t, states[t])
        running = ______      # ⭐ optional: this step's delta, plus the running total carried back
        adv[t]  = running
    return adv

# --- check the two claims in the table above, on a real campaign
np.random.seed(7); torch.manual_seed(7)
st, ac, rw, nx, _ = sample_episode(theta_pi)

one_step = [rw[t] + GAMMA*value_of(V_PI, t+1, nx[t]) - value_of(V_PI, t, st[t]) for t in range(N_DAYS)]
G        = returns_to_go(rw, GAMMA)
full_adv = [G[t] - value_of(V_PI, t, st[t]) for t in range(N_DAYS)]

assert np.allclose(gae_advantages(rw, st, nx, V_PI, lam=0.0), one_step), "λ=0 should be notebook 03's δ."
assert np.allclose(gae_advantages(rw, st, nx, V_PI, lam=1.0), full_adv), "λ=1 should be notebook 02's advantage."
print("✅ λ = 0  →  exactly the one-step advantage (notebook 03)")
print("✅ λ = 1  →  exactly the full-return advantage (notebook 02)")
print("   Everything in between is a blend nobody had to pick an integer for.")

In [ ]:
#@title ⭐ OPTIONAL — the same variance story, now continuous in λ  { display-mode: "form" }
np.random.seed(1); torch.manual_seed(1)
episodes = [sample_episode(theta_pi) for _ in range(1500)]

lams, variances = np.linspace(0, 1, 11), []
for lam in lams:
    pool = []
    for st, ac, rw, nx, _ in episodes:
        pool += list(gae_advantages(rw, st, nx, V_PI, lam=lam))
    variances.append(np.var(pool))

plt.figure(figsize=(7.5, 4.2))
plt.plot(lams, variances, "o-", color="#9467bd", lw=2)
plt.scatter([0], [variances[0]],  color="#4a5bd0", s=70, zorder=5, label="λ=0 · notebook 03")
plt.scatter([1], [variances[-1]], color="#dd8452", s=70, zorder=5, label="λ=1 · notebook 02")
plt.xlabel("GAE λ"); plt.ylabel("variance of the advantage")
plt.title("One continuous dial between the two algorithms you have written")
plt.legend(); plt.tight_layout(); plt.show()
print("Same trade as Part 1, now with no integers in it — and this is where the next exercise starts.")

---
# 🎓 Wrap-up

| The idea | The formula | In one line |
|---|---|---|
| **n-step return** | `r_t + … + γⁿ⁻¹r_{t+n-1} + γⁿV(s_{t+n})` | keep `n` real days, then let the critic finish |
| **n-step advantage** | `G⁽ⁿ⁾_t − V(s_t)` | the weight the actor is handed |
| **`n = 1`** | `r + γV(s′) − V(s)` | notebook 03's A2C |
| **`n = ∞`** | `G_t − V(s_t)` | notebook 02's REINFORCE-with-a-baseline |
| **The trade** | short `n` ⇒ less variance, more bias | how far you trust the critic |

**The three things worth carrying out of here**

1. **Notebooks 02 and 03 were never two algorithms.** They are the two ends of one dial, and you wrote
   the function that contains both — `n_step_target`, checked against each of them by assertion in
   Task 1.
2. **The dial trades two measurable things, and you measured both.** Variance falls as `n` shrinks
   (Part 1); the damage from a wrong critic grows as `n` shrinks (Part 2.1). Neither is a matter of
   opinion, and neither is visible if you only ever look at the learning curve.
3. **The right `n` is a statement about your problem, not about RL.** Our three-day campaign is too
   short for the left end of the dial to pay for itself — a fact about the toy. Horizon length and
   critic quality decide it, every time.

### Where this goes next
- **Next exercise — GAE & PPO:** stop picking an integer (**GAE**, previewed above), and stop throwing
  a batch away after one update (**PPO**).
- **RL for LLMs:** the same actor and critic, where the campaign is a generated answer, the moves are
  tokens, and the reward comes from a preference model.